## Working directory and import libraries

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import json
import cv2
import torch
import ultralytics
from pathlib import Path

PROJECT_ROOT = "/home/user/NeuroVision-AI-2"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
print("Working directory:", os.getcwd())

# import cfgs
from src.config import cfg

#  Quick check
print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("ultralytics :", ultralytics.__version__)
print("opencv      :", cv2.__version__)

print("Raw data :", cfg.RAW_DIR)
print("Classes  :", cfg.CLASS_NAMES)
print("Split    : train %.0f%% / val %.0f%% / test %.0f%%" % (
    (1 - cfg.VAL_RATIO - cfg.TEST_RATIO) * 100, cfg.VAL_RATIO * 100, cfg.TEST_RATIO * 100))
print("Seed     :", cfg.SEED)

Working directory: /home/user/NeuroVision-AI-2
python      : 3.10.22
torch       : 2.14.1+cu130 | CUDA: False
ultralytics : 8.4.192
opencv      : 5.0.0
Raw data : /home/user/NeuroVision-AI-2/data/raw
Classes  : ('Glioma', 'Meningioma', 'No Tumor', 'Pituitary')
Split    : train 70% / val 15% / test 15%
Seed     : 42


/venv/main/lib/python3.10/site-packages/torch/cuda/__init__.py:228: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## Configuration and MLflow connection

In [5]:
from src.config import cfg
from src.mlflow_utils import start_tracking

start_tracking(experiment="neurovision-yolov8")
print(json.dumps(cfg.to_dict(), indent=2)[:600], "...")

MLflow tracking: http://127.0.0.1:5000 | experiment: neurovision-yolov8 (id 1)
{
  "SEED": 42,
  "PROJECT_ROOT": "/home/user/NeuroVision-AI-2",
  "RAW_DIR": "/home/user/NeuroVision-AI-2/data/raw",
  "DATA_DIR": "/home/user/NeuroVision-AI-2/data",
  "DATA_YAML": "/home/user/NeuroVision-AI-2/data/data.yaml",
  "MODELS_DIR": "/home/user/NeuroVision-AI-2/models",
  "RUNS_DIR": "/home/user/NeuroVision-AI-2/runs",
  "VAL_RATIO": 0.15,
  "TEST_RATIO": 0.15,
  "RAW_SPLIT_DIR": "/home/user/NeuroVision-AI-2/data/raw_split",
  "CLASS_NAMES": [
    "Glioma",
    "Meningioma",
    "No Tumor",
    "Pituitary"
  ],
  "IMAGE_SIZE": 640,
  "BATCH_SIZE": 16,
  "EPOCHS": 30,
  "PATIENCE":  ...


## Check the Vast GPU

In [7]:
from src.train import check_gpu

gpu = check_gpu()
print(json.dumps(gpu, indent=2))

{
  "cuda_available": false,
  "torch_version": "2.14.1+cu130",
  "cuda_version": "13.0",
  "device_count": 0,
  "device_name": null,
  "total_memory_gb": null,
  "free_memory_gb": null
}


## Augmentation: settings and visual check

In [ ]:
from src.config import AUGMENTATION
from src.train import preview_augmentation

print(json.dumps(AUGMENTATION, indent=2))
preview_augmentation(n=6)       # augmented training images with their boxes

## One-epoch test

In [ ]:
from src.train import train_yolo

smoke = train_yolo(model_name="yolov8n.pt", epochs=1, run_name="smoke-test", mlflow_log=False)
print(smoke["run_dir"], "| duration: %.0f s" % smoke["duration_s"])

## Full training, baseline

In [ ]:
baseline = train_yolo(model_name="yolov8n.pt", run_name="yolov8n-baseline")
print("run dir  :", baseline["run_dir"])
print("duration : %.1f min" % (baseline["duration_s"] / 60))
print(json.dumps(baseline["metrics"], indent=2))

## Read the curves and detect overfitting

In [ ]:
from src.visualization import plot_training_curves

plot_training_curves(baseline["run_dir"])

## Second run, one setting changed

In [ ]:
second = train_yolo(model_name="yolov8s.pt", run_name="yolov8s")
print("duration : %.1f min" % (second["duration_s"] / 60))
print(json.dumps(second["metrics"], indent=2))

## Compare runs and register the champion

In [ ]:
from src.mlflow_utils import compare_runs, register_champion

runs = compare_runs(experiment="neurovision-yolov8")     # DataFrame: params, recall, precision, mAP, duration
display(runs)